# Figure 4.2 — Study Area Maps
### Research Stakeholders and Data Sources

This notebook produces the four zoomed study-area maps used in **Section 4.2** of the thesis:

1. **Sri Lanka** (target domain) — Kandy, Kegalle, Nuwara Eliya, Badulla districts
2. **Kerala, India** (source domain)
3. **Thailand** (source domain)
4. **Indonesia** (source domain)

Each map is zoomed to the **actual spatial extent of the landslide observations** in each dataset (rather than showing the whole country), and a combined 4-panel figure is produced at the end for direct inclusion in the thesis.

> **Prerequisite:** This notebook assumes you already have a `datasets` dictionary (e.g. `datasets["Sri Lanka"]`, `datasets["Kerala"]`, `datasets["Thailand"]`, `datasets["Indonesia"]`) loaded from your earlier data-preparation steps, each containing `x`/`y` (or `longitude`/`latitude`) columns. Update the loading cell below if your variable names differ.

## 1. Setup

In [5]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path

# Output directory for figures
OUTPUT_DIR = Path("outputs/figures")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.dpi"] = 150


### 1.1 Load your datasets

If `datasets` is not already in memory from a previous notebook, load it here. Replace the paths below with your actual file locations.

In [6]:
# --- Adjust this cell to match how your data is actually stored ---
# Example (uncomment and edit as needed):
#
datasets = {
    "Sri Lanka": pd.read_csv("../data/processed/training_table_sri_lanka_selected_districts_base.csv"),
    "Kerala": pd.read_csv("../data/processed/training_table_kerala_extended.csv"),
    "Thailand": pd.read_csv("../data/processed/thailand_landslide_event_level_with_factors.csv"),
    "Indonesia": pd.read_csv("../data/processed/training_table_indonesia_extended.csv"),
}

assert "datasets" in globals(), "Load your `datasets` dict (Sri Lanka, Kerala, Thailand, Indonesia) before continuing."
print(list(datasets.keys()))


['Sri Lanka', 'Kerala', 'Thailand', 'Indonesia']


### 1.2 Load a world country-boundary layer

Used for the Thailand and Indonesia country outlines.

In [8]:
import geopandas as gpd

url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(url)
world.head()

,featurecla,scalerank,LABELRANK,SOVEREIGNT,SOV_A3,ADM0_DIF,LEVEL,TYPE,TLC,ADMIN,...,FCLASS_TR,FCLASS_ID,FCLASS_PL,FCLASS_GR,FCLASS_IT,FCLASS_NL,FCLASS_SE,FCLASS_BD,FCLASS_UA,geometry
0,Admin-0 country,1,6,Fiji,FJI,0,2,Sovereign country,1,Fiji,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"MULTIPOLYGON (((180 -16.06713, 180 -16.55522, ..."
1,Admin-0 country,1,3,United Republic of Tanzania,TZA,0,2,Sovereign country,1,United Republic of Tanzania,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((33.90371 -0.95, 34.07262 -1.05982, 3..."
2,Admin-0 country,1,7,Western Sahara,SAH,0,2,Indeterminate,1,Western Sahara,...,Unrecognized,Unrecognized,Unrecognized,NaN,NaN,Unrecognized,NaN,NaN,NaN,"POLYGON ((-8.66559 27.65643, -8.66512 27.58948..."
3,Admin-0 country,1,2,Canada,CAN,0,2,Sovereign country,1,Canada,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"MULTIPOLYGON (((-122.84 49, -122.97421 49.0025..."
4,Admin-0 country,1,2,United States of America,US1,1,2,Country,1,United States of America,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"MULTIPOLYGON (((-122.84 49, -120 49, -117.0312..."


## 2. Check coordinate ranges (confirm x/y are longitude/latitude)

In [9]:
for country, df in datasets.items():
    print("\n", country)
    print(
        "X:", df["x"].min(), "to", df["x"].max()
    )
    print(
        "Y:", df["y"].min(), "to", df["y"].max()
    )



 Sri Lanka
X: 408303.8917984216 to 526474.8942286703
Y: 728397.0992466612 to 837600.8984391758

 Kerala


KeyError: 'x'

Expected broad ranges for reference:

```
Sri Lanka   X: ~79 - 82    Y: ~5 - 10
Kerala      X: ~74 - 78    Y: ~8 - 13
Thailand    X: ~97 - 106   Y: ~5 - 21
Indonesia   X: ~95 - 141   Y: ~-11 - 6
```

Your dataset will likely cover only a subset of these ranges — that's expected, and is exactly what the zoom function below uses.

## 3. Reusable zoom-map function

In [ ]:
def create_zoom_map(df, country_boundary, title, output_name, padding=0.5):
    """Plot a country boundary zoomed to the actual extent of the dataset's points."""

    x = pd.to_numeric(df["x"], errors="coerce")
    y = pd.to_numeric(df["y"], errors="coerce")

    valid = x.notna() & y.notna()
    x = x[valid]
    y = y[valid]

    fig, ax = plt.subplots(figsize=(7, 7))

    # Country boundary
    country_boundary.plot(
        ax=ax, facecolor="white", edgecolor="black", linewidth=0.8
    )

    # Actual dataset locations
    ax.scatter(
        x, y, s=10, facecolor="black", edgecolor="none",
        alpha=0.6, label="Landslide observations"
    )

    # Zoom automatically to actual dataset extent
    ax.set_xlim(x.min() - padding, x.max() + padding)
    ax.set_ylim(y.min() - padding, y.max() + padding)

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title(title, fontsize=12)
    ax.legend(loc="best", frameon=True)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / output_name, dpi=300, bbox_inches="tight")
    plt.show()

    return fig, ax


## 4. Indonesia — source-domain study area

In [ ]:
indonesia_boundary = world[world["ADMIN"] == "Indonesia"]

create_zoom_map(
    datasets["Indonesia"],
    indonesia_boundary,
    "Indonesia Source Study Area",
    "Figure_Indonesia_Study_Area.png",
    padding=0.5
)


> **Figure 4.2(a): Indonesian Source-Domain Study Area and Landslide Observations**

## 5. Thailand — source-domain study area

In [ ]:
thailand_boundary = world[world["ADMIN"] == "Thailand"]

create_zoom_map(
    datasets["Thailand"],
    thailand_boundary,
    "Thailand Source Study Area",
    "Figure_Thailand_Study_Area.png",
    padding=0.5
)


> **Figure 4.2(b): Thailand Source-Domain Study Area and Landslide Observations**

## 6. Kerala — source-domain study area

Kerala needs an Indian **state-level (admin-1)** boundary rather than the national outline, so we pull a GADM layer and select Kerala specifically.

In [ ]:
kerala_url = (
    "https://geodata.ucdavis.edu/"
    "gadm/gadm4.1/json/"
    "gadm41_IND_1.json"
)

india_states = gpd.read_file(kerala_url)
print(india_states["NAME_1"].unique())


In [ ]:
kerala_boundary = india_states[
    india_states["NAME_1"] == "Kerala"
].copy()

create_zoom_map(
    datasets["Kerala"],
    kerala_boundary,
    "Kerala Source Study Area",
    "Figure_Kerala_Study_Area.png",
    padding=0.25
)


> **Figure 4.2(c): Kerala Source-Domain Study Area and Landslide Observations**

## 7. Sri Lanka — target-domain study area

Sri Lanka gets the most detailed treatment: **district-level (admin-2)** boundaries, restricted to the four districts actually used in the research — **Kandy, Kegalle, Nuwara Eliya, and Badulla**.

In [ ]:
sri_lanka_url = (
    "https://geodata.ucdavis.edu/"
    "gadm/gadm4.1/json/"
    "gadm41_LKA_2.json"
)

sl_districts = gpd.read_file(sri_lanka_url)
print(sorted(sl_districts["NAME_2"].unique()))


In [ ]:
target_districts = [
    "Kandy",
    "Kegalle",
    "Nuwara Eliya",
    "Badulla"
]

selected_districts = sl_districts[
    sl_districts["NAME_2"].isin(target_districts)
].copy()

selected_districts


In [ ]:
fig, ax = plt.subplots(figsize=(8, 9))

# All districts (context, light gray outline)
sl_districts.plot(
    ax=ax, facecolor="white", edgecolor="lightgray", linewidth=0.5
)

# Selected study districts (highlighted)
selected_districts.plot(
    ax=ax, facecolor="lightgray", edgecolor="black", linewidth=1
)

# District name labels
for _, row in selected_districts.iterrows():
    point = row.geometry.representative_point()
    ax.text(
        point.x, point.y, row["NAME_2"],
        fontsize=8, ha="center", fontweight="bold"
    )


In [ ]:
sl = datasets["Sri Lanka"].copy()

longitude = pd.to_numeric(sl["longitude"], errors="coerce")
latitude = pd.to_numeric(sl["latitude"], errors="coerce")

valid = longitude.notna() & latitude.notna()

ax.scatter(
    longitude[valid], latitude[valid],
    s=10, facecolor="black", edgecolor="none",
    alpha=0.6, label="Landslide observations"
)

fig


In [ ]:
xmin, ymin, xmax, ymax = selected_districts.total_bounds
padding = 0.15

ax.set_xlim(xmin - padding, xmax + padding)
ax.set_ylim(ymin - padding, ymax + padding)

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Sri Lankan Target Study Areas", fontsize=12)
ax.legend(loc="best")

plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "Figure_Sri_Lanka_Target_Study_Area.png",
    dpi=300, bbox_inches="tight"
)
plt.show()


> **Figure 4.2(d): Sri Lankan Target-Domain Study Areas and Landslide Observations**

## 8. Combined 4-panel Figure 4.2

Rather than four separate full-page figures, assemble a single figure with four panels — (a) Indonesia, (b) Thailand, (c) Kerala, (d) Sri Lanka — for one clean insertion into the thesis.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 14))

panel_specs = [
    ("Indonesia", indonesia_boundary, "(a) Indonesia", axes[0, 0]),
    ("Thailand",  thailand_boundary,  "(b) Thailand",  axes[0, 1]),
    ("Kerala",    kerala_boundary,    "(c) Kerala, India", axes[1, 0]),
]

for country, boundary, label, ax in panel_specs:
    df = datasets[country]
    x = pd.to_numeric(df["x"], errors="coerce")
    y = pd.to_numeric(df["y"], errors="coerce")
    valid = x.notna() & y.notna()
    x, y = x[valid], y[valid]

    boundary.plot(ax=ax, facecolor="white", edgecolor="black", linewidth=0.8)
    ax.scatter(x, y, s=8, facecolor="black", edgecolor="none", alpha=0.6)

    padding = 0.25 if country == "Kerala" else 0.5
    ax.set_xlim(x.min() - padding, x.max() + padding)
    ax.set_ylim(y.min() - padding, y.max() + padding)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title(label, fontsize=12)

# Panel (d): Sri Lanka
ax = axes[1, 1]
sl_districts.plot(ax=ax, facecolor="white", edgecolor="lightgray", linewidth=0.5)
selected_districts.plot(ax=ax, facecolor="lightgray", edgecolor="black", linewidth=1)

for _, row in selected_districts.iterrows():
    point = row.geometry.representative_point()
    ax.text(point.x, point.y, row["NAME_2"], fontsize=7, ha="center", fontweight="bold")

ax.scatter(longitude[valid], latitude[valid], s=8, facecolor="black", edgecolor="none", alpha=0.6)

xmin, ymin, xmax, ymax = selected_districts.total_bounds
padding = 0.15
ax.set_xlim(xmin - padding, xmax + padding)
ax.set_ylim(ymin - padding, ymax + padding)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("(d) Sri Lanka", fontsize=12)

fig.suptitle(
    "Figure 4.2: Detailed Geographical Distribution of the Source and Target Study Areas",
    fontsize=14, y=1.02
)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "Figure_4_2_Combined_Study_Areas.png", dpi=300, bbox_inches="tight")
plt.show()


> **Figure 4.2: Detailed Geographical Distribution of the Source and Target Study Areas: (a) Indonesia, (b) Thailand, (c) Kerala, India, and (d) Sri Lanka.**

All individual panel images and the combined figure are saved to `outputs/figures/`:

- `Figure_Indonesia_Study_Area.png`
- `Figure_Thailand_Study_Area.png`
- `Figure_Kerala_Study_Area.png`
- `Figure_Sri_Lanka_Target_Study_Area.png`
- `Figure_4_2_Combined_Study_Areas.png` (use this one in the thesis)